# 05 · Practical C — The Black-Scholes Option Pricer

**Read first:** Chapter 5 (the formula section), then Practical C.

---

## What you'll be able to do after this

- Price a European vanilla in one line with Garman–Kohlhagen, and convert the answer into every unit a desk actually quotes.
- Compute delta and vega two independent ways — closed form and finite difference — and say when you'd reach for each.
- Explain why put–call parity needs a discount factor, and what breaks if you leave it out.

## The intuition, before the maths

In notebook 04 you priced an option by chopping the future into 101 buckets, weighting each payoff by its probability, and adding them up. It worked, and it was slow.

Black-Scholes is the same calculation with the sum done **once, on paper, algebraically**. Somebody worked out the integral in closed form so nobody has to compute it numerically ever again. That's genuinely all this is.

### What the formula is saying

The call price has two terms:

$$\text{call} = \underbrace{S\,e^{-r_1 T}N(d_1)}_{\text{what you receive}} - \underbrace{K\,e^{-r_2 T}N(d_2)}_{\text{what you pay}}$$

Read it as a trade you only do when it's worth doing:

- $N(d_2)$ is, near enough, **the probability you end up exercising**. Multiply by the strike and discount: that's the expected cost of buying at the strike.
- $N(d_1)$ is the same idea for the asset side, but probability-weighted by *how much* spot you receive, not just whether you receive any.

The difference is what the option is worth.

### The thing that surprises people

The formula contains **no view on where spot is going**. There's no expected return, no drift you chose, nothing about whether you're bullish. Just spot, strike, time, rates and volatility.

That's not an oversight. The derivation assumes you continuously delta hedge at no cost, which removes every source of risk *except* volatility. Once direction is hedged away, your opinion about direction stops being worth anything. What's left to price is the wiggle.

Chapter 2 puts the practical consequence bluntly: the FX derivatives market quotes vanillas in **volatility** terms and uses Black-Scholes purely as the translation layer into a cash premium. The formula is a unit converter that the whole market agrees on.

## The maths, derived not asserted

### The forward

Set volatility to zero in the Chapter 5 SDE and spot follows a deterministic path:

$$F_T = S\,e^{(r_{CCY2} - r_{CCY1})T}$$

Only the **differential** matters. Both rates at 5% gives the same forward as both at 0%.

### Garman–Kohlhagen

The FX extension of Black-Scholes — one discount rate per currency, since both legs of an FX trade earn interest:

$$\text{call} = S e^{-r_1 T}N(d_1) - K e^{-r_2 T}N(d_2)$$
$$\text{put} = K e^{-r_2 T}N(-d_2) - S e^{-r_1 T}N(-d_1)$$

$$d_1 = \frac{\ln(S/K) + \left(r_2 - r_1 + \frac{\sigma^2}{2}\right)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

Where the pieces come from, so none of it is arbitrary:

- $\ln(S/K)$ — how far the strike is from spot, **in log space**, because that's the space the returns are normal in.
- $(r_2 - r_1)T$ — the drift that carries spot to the forward.
- $+\sigma^2/2$ — the Itō correction from notebook 04, back again.
- $\div\ \sigma\sqrt{T}$ — divide by the standard deviation, converting a distance into *a number of standard deviations*. That's why $N(\cdot)$ can then be applied: it wants a z-score.
- $d_2 = d_1 - \sigma\sqrt{T}$ — the same distance measured from the other end of the distribution.

### Put–call parity, and the trap in it

At maturity, long call + short put = long forward at the strike, whatever spot does. So the prices must satisfy:

$$\text{call} - \text{put} = e^{-r_{CCY2}T}(F - K)$$

The book first writes this as `call − put = F − K`, then immediately shows why that's wrong: **option prices are present valued, but the `F − K` difference is a P&L you realise in the future.** You have to discount it to compare like with like.

The undiscounted version only coincides when $r_2 = 0$ or $K = F$. Experiment 2 breaks it deliberately.

### First-order greeks

$$\Delta_{\text{call}} = e^{-r_1 T}N(d_1), \qquad \Delta_{\text{put}} = e^{-r_1 T}\left[N(d_1) - 1\right]$$
$$\nu = S e^{-r_1 T} n(d_1)\sqrt{T}$$

with $n$ the normal density. Two things fall straight out:

- The two deltas differ by exactly $e^{-r_1 T}$ — **put–call parity in greek terms.** A call becomes a put by selling the forward.
- Vega has no call/put version. A forward has no volatility exposure, so calls and puts at the same strike share their vega *and* their gamma. This is why traders talk about **strikes**, not calls and puts, once a position is in the book (Ch. 6).

## The code

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from fxds.conventions import (
    OptionType, CurrencyPair,
    ccy2_pips_to_ccy2_cash, ccy2_cash_to_ccy1_cash, ccy2_pips_to_ccy1_pct,
)
from fxds.blackscholes import (
    forward, price, d1_d2, put_call_parity_rhs,
    delta_closed_form, delta_finite_difference,
    vega_closed_form, vega_market, vega_finite_difference,
    gamma_closed_form,
)
from fxds.plotting import (
    use_house_style, style_axis, mark_level, as_percent, style_plotly,
    PRIMARY, SECONDARY, TERTIARY, QUATERNARY, MUTED, ALERT, CALL_COLOUR, PUT_COLOUR,
)

use_house_style()
pd.set_option("display.precision", 6)

### Task A, Step 1 — the forward

Check what happens when the two rates are equal.

In [2]:
rows = []
for r1, r2 in [(0.00, 0.00), (0.05, 0.05), (0.00, 0.05), (0.05, 0.00), (0.02, 0.06)]:
    rows.append({
        "r_ccy1": r1, "r_ccy2": r2, "differential": r2 - r1,
        "forward (1y)": forward(1.3000, 1.0, r1, r2),
    })
pd.DataFrame(rows)

,r_ccy1,r_ccy2,differential,forward (1y)
0,0.00,0.00,0.00,1.300000
1,0.05,0.05,0.00,1.300000
2,0.00,0.05,0.05,1.366652
3,0.05,0.00,-0.05,1.236598
4,0.02,0.06,0.04,1.353054


Equal rates pin the forward to spot regardless of *level* — only the differential moves it. Higher CCY2 rates give **positive drift** (forward above spot); higher CCY1 rates give **negative drift**.

### Task A, Step 2 — the pricer, and the book's acceptance test

In [3]:
BOOK = dict(spot=1.0, strike=1.0, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)

call = price(OptionType.CALL, **BOOK)
put = price(OptionType.PUT, **BOOK)
d1, d2 = d1_d2(**BOOK)

print(f"S = K = 1.0,  T = 1.0,  sigma = 10%,  r1 = r2 = 0%")
print(f"  d1              {d1:+.6f}")
print(f"  d2              {d2:+.6f}")
print(f"  call price      {call:.6f} CCY2 pips     ← book: 0.0399 ✓")
print(f"  put price       {put:.6f} CCY2 pips")
print(f"  call == put?    {np.isclose(call, put)}  (strike sits at the forward here)")

S = K = 1.0,  T = 1.0,  sigma = 10%,  r1 = r2 = 0%
  d1              +0.050000
  d2              -0.050000
  call price      0.039878 CCY2 pips     ← book: 0.0399 ✓
  put price       0.039878 CCY2 pips
  call == put?    True  (strike sits at the forward here)


### Task A, Step 2 — the three worked examples

**Example 1:** move the strike on a call.

In [4]:
base = dict(spot=100.0, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)
rows = [{"strike": k,
         "call": price(OptionType.CALL, strike=k, **base),
         "put":  price(OptionType.PUT,  strike=k, **base)}
        for k in [90.0, 95.0, 100.0, 105.0, 110.0]]
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nStrike up → call cheapens (payoff moves away from the forward), put richens.")

 strike      call       put
   90.0 10.712381  0.712381
   95.0  6.888063  1.888063
  100.0  3.987761  3.987761
  105.0  2.064019  7.064019
  110.0  0.953947 10.953947

Strike up → call cheapens (payoff moves away from the forward), put richens.


**Example 2:** raise volatility, then time. Watch *both* sides.

In [5]:
rows = []
for sigma in [0.05, 0.10, 0.20, 0.40]:
    a = dict(spot=1.25, strike=1.25, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=sigma)
    rows.append({"sigma": sigma, "T": 1.0,
                 "call": price(OptionType.CALL, **a), "put": price(OptionType.PUT, **a)})
for T in [0.25, 1.0, 4.0]:
    a = dict(spot=1.25, strike=1.25, T=T, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)
    rows.append({"sigma": 0.10, "T": T,
                 "call": price(OptionType.CALL, **a), "put": price(OptionType.PUT, **a)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nBoth columns rise in both blocks: a wider distribution brings bigger payoffs into play")
print("on BOTH sides. Volatility is not directional.")

 sigma    T     call      put
  0.05 1.00 0.024931 0.024931
  0.10 1.00 0.049847 0.049847
  0.20 1.00 0.099570 0.099570
  0.40 1.00 0.198149 0.198149
  0.10 0.25 0.024931 0.024931
  0.10 1.00 0.049847 0.049847
  0.10 4.00 0.099570 0.099570

Both columns rise in both blocks: a wider distribution brings bigger payoffs into play
on BOTH sides. Volatility is not directional.


**Example 3:** raise both rates together — forward unchanged, prices fall.

In [6]:
rows = []
for r in [0.00, 0.03, 0.06, 0.10]:
    a = dict(spot=1.0, strike=1.0, T=1.0, r_ccy1=r, r_ccy2=r, sigma=0.10)
    rows.append({"r1 = r2": r, "forward": forward(1.0, 1.0, r, r),
                 "call": price(OptionType.CALL, **a),
                 "put": price(OptionType.PUT, **a),
                 "discount factor": np.exp(-r * 1.0)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe forward is pinned. The distribution never moves. Only the discounting changes.")

 r1 = r2  forward     call      put  discount factor
    0.00      1.0 0.039878 0.039878         1.000000
    0.03      1.0 0.038699 0.038699         0.970446
    0.06      1.0 0.037555 0.037555         0.941765
    0.10      1.0 0.036083 0.036083         0.904837

The forward is pinned. The distribution never moves. Only the discounting changes.


### Task A, Step 3 — notionals and the unit chain

Prices come out of the formula in **CCY2 pips** — CCY2 per one CCY1. Desks quote in several other units, and the conversions are one multiply each:

```
CCY2 pips  ×  CCY1 notional  =  CCY2 cash
CCY2 cash  ÷  spot           =  CCY1 cash
CCY2 pips  ÷  spot           =  CCY1%
```

In [7]:
pair = CurrencyPair.parse("EURUSD")
spot, strike, T, r1, r2, sigma = 1.3000, 1.3200, 0.5, 0.005, 0.025, 0.085
notional_eur = 10_000_000

pips = price(OptionType.CALL, spot, strike, T, r1, r2, sigma)
usd_cash = ccy2_pips_to_ccy2_cash(pips, notional_eur)
eur_cash = ccy2_cash_to_ccy1_cash(usd_cash, spot)
eur_pct = ccy2_pips_to_ccy1_pct(pips, spot)

print(f"{pair}  {strike} EUR call / USD put,  6 month,  EUR{notional_eur:,} notional")
print(f"  price          {pips:.6f} USD pips  ({pips / pair.pip:.1f} pips)")
print(f"  USD cash       {usd_cash:>14,.2f}")
print(f"  EUR cash       {eur_cash:>14,.2f}")
print(f"  as EUR%        {eur_pct:.4%}   ({eur_pct / 0.0001:.1f} basis points)")

EUR/USD  1.32 EUR call / USD put,  6 month,  EUR10,000,000 notional
  price          0.027866 USD pips  (278.7 pips)
  USD cash           278,663.35
  EUR cash           214,356.42
  as EUR%        2.1436%   (214.4 basis points)


### Task A, Step 4 — put–call parity, and why the naive version fails

This is the part of the practical that teaches something rather than just checking arithmetic.

In [8]:
spot, strike, T, r1, r2, sigma = 1.30, 1.20, 2.0, 0.01, 0.06, 0.12

c = price(OptionType.CALL, spot, strike, T, r1, r2, sigma)
p = price(OptionType.PUT,  spot, strike, T, r1, r2, sigma)
F = forward(spot, T, r1, r2)

correct = put_call_parity_rhs(spot, strike, T, r1, r2)
naive = put_call_parity_rhs(spot, strike, T, r1, r2, discounted=False)

print(f"forward to 2y            {F:.6f}")
print(f"call - put               {c - p:+.8f}")
print()
print(f"e^(-r2·T)·(F - K)        {correct:+.8f}   ← matches ✓")
print(f"        (F - K)          {naive:+.8f}   ← wrong by {abs(naive - correct):.6f}")
print()
print(f"the discrepancy is exactly the discount factor: {correct / naive:.8f} = e^(-r2·T) = {np.exp(-r2*T):.8f}")

forward to 2y            1.436722
call - put               +0.20995375

e^(-r2·T)·(F - K)        +0.20995375   ← matches ✓
        (F - K)          +0.23672219   ← wrong by 0.026768

the discrepancy is exactly the discount factor: 0.88692044 = e^(-r2·T) = 0.88692044


**Why.** Both option premiums are paid *today*, so both are present values. But `F − K` is a P&L you collect at *maturity*. Comparing a present value against a future value is a units error, no different from adding dollars to euros. Discounting `F − K` puts them in the same units.

The naive version coincides in exactly two cases — and it's worth seeing where it stops mattering.

In [9]:
fig, ax = plt.subplots()
strikes = np.linspace(1.05, 1.55, 200)
spot, T, r1, r2, sigma = 1.30, 2.0, 0.01, 0.06, 0.12
F = forward(spot, T, r1, r2)

actual = [price(OptionType.CALL, spot, k, T, r1, r2, sigma)
          - price(OptionType.PUT, spot, k, T, r1, r2, sigma) for k in strikes]
naive_line = [put_call_parity_rhs(spot, k, T, r1, r2, discounted=False) for k in strikes]

ax.plot(strikes, actual, color=PRIMARY, linewidth=2.5, label="call − put (actual)")
ax.plot(strikes, naive_line, color=ALERT, linewidth=2, linestyle="--",
        label="F − K (undiscounted, wrong)")
ax.axhline(0, color=MUTED, linewidth=0.8)
mark_level(ax, F, "forward")

ax.legend()
style_axis(
    ax,
    "Put–call parity: the undiscounted version is wrong everywhere except one point",
    "Strike (CCY2 per CCY1)",
    "call − put (CCY2 pips)",
    "The two lines cross only where K = F, and the gap widens with distance. The slope differs too — by the discount factor.",
)
plt.show()

### Task B — the pricing function, and the guard

Task B in the book is about moving from spreadsheet cells to a VBA function. In Python that step doesn't exist — the module function *is* the implementation. What does carry over is the **guard**: the book asks for an explicit check on non-positive time or volatility, clamping them to a tiny positive value so the formula returns the payoff at maturity instead of dividing by zero.

> The book's printed VBA reads `If (T >= 0) Then T = 0.0000000001`, which fires on *every valid input* and destroys the price. That's a transcription defect — the surrounding prose states the intent plainly. Implemented here as `<=`, and recorded in `notes/deviations.md`. There's a regression test for it in `tests/test_blackscholes.py`.

In [10]:
print("Expired options collapse to their payoff:")
for S in [0.90, 1.00, 1.10]:
    c = price(OptionType.CALL, S, 1.0, 0.0, 0.0, 0.0, 0.10)
    p = price(OptionType.PUT,  S, 1.0, 0.0, 0.0, 0.0, 0.10)
    print(f"  spot {S:.2f}   call {c:.6f}  (payoff {max(S-1,0):.2f})"
          f"   put {p:.6f}  (payoff {max(1-S,0):.2f})")

print("\nZero volatility → spot follows the forward exactly, so the option is worth")
print("its discounted payoff against the forward, not against spot:")
S, T, r1, r2 = 1.0, 1.0, 0.0, 0.05
print(f"  forward {forward(S, T, r1, r2):.6f}"
      f"   call struck 1.00 = {price(OptionType.CALL, S, 1.0, T, r1, r2, 0.0):.6f}")

Expired options collapse to their payoff:
  spot 0.90   call 0.000000  (payoff 0.00)   put 0.100000  (payoff 0.10)
  spot 1.00   call 0.000000  (payoff 0.00)   put 0.000000  (payoff 0.00)
  spot 1.10   call 0.100000  (payoff 0.10)   put 0.000000  (payoff 0.00)

Zero volatility → spot follows the forward exactly, so the option is worth
its discounted payoff against the forward, not against spot:
  forward 1.051271   call struck 1.00 = 0.048771


### Task C — greeks, two ways

In [11]:
delta_cf = delta_closed_form(OptionType.CALL, **BOOK)
vega_mkt = vega_market(**BOOK)

print(f"S = K = 1.0,  T = 1.0,  sigma = 10%,  zero rates")
print(f"  delta (closed form)   {delta_cf:.4%}    ← book: 'close to 50%' ✓")
print(f"  vega  (market terms)  {vega_mkt:.5%}   ← book: 'a shade under 0.40%' ✓")
print()
print(f"  put delta             {delta_closed_form(OptionType.PUT, **BOOK):+.4%}")
print(f"  call − put delta      {delta_cf - delta_closed_form(OptionType.PUT, **BOOK):.6f}"
      f"  = e^(-r1·T) = {np.exp(-0.0):.6f}")

S = K = 1.0,  T = 1.0,  sigma = 10%,  zero rates
  delta (closed form)   51.9939%    ← book: 'close to 50%' ✓
  vega  (market terms)  0.39844%   ← book: 'a shade under 0.40%' ✓

  put delta             -48.0061%
  call − put delta      1.000000  = e^(-r1·T) = 1.000000


> **Why 52% and not 50%?** With `S = K`, `d₁ = σ√T/2 = 0.05`, which is slightly positive — so `N(d₁)` is slightly above a half. The strike sits at *spot*, but the delta-neutral point sits at the **forward adjusted by the Itō term** (Ch. 8). The book says "close to 50%" rather than "50%" for exactly this reason. This is also why "ATM" meaning *delta-neutral straddle* is not the same as *at-the-forward* — see the misconceptions below.

Now the same numbers by finite difference: bump the input, reprice, divide.

In [12]:
args = dict(spot=1.30, strike=1.25, T=2.0, r_ccy1=0.03, r_ccy2=0.01, sigma=0.15)

rows = []
for ot in (OptionType.CALL, OptionType.PUT):
    rows.append({
        "option": ot.value,
        "delta (closed)": delta_closed_form(ot, **args),
        "delta (finite)": delta_finite_difference(ot, **args, spot_flex=1e-6),
        "vega (closed)": vega_market(**args),
        "vega (finite)": vega_finite_difference(ot, **args, vol_flex=1e-6),
    })
out = pd.DataFrame(rows)
print(out.to_string(index=False, float_format=lambda v: f"{v:+.9f}"))
print("\nBoth methods agree to nine figures. Note the two vega rows are identical:")
print("a forward has no volatility exposure, so calls and puts share vega (Ch. 6).")

option  delta (closed)  delta (finite)  vega (closed)  vega (finite)
  call    +0.509284977    +0.509284977   +0.005285557   +0.005285557
   put    -0.432479556    -0.432479556   +0.005285557   +0.005285557

Both methods agree to nine figures. Note the two vega rows are identical:
a forward has no volatility exposure, so calls and puts share vega (Ch. 6).


**Which to use.** Closed form is faster and exact — but it only exists if somebody has done the differentiation, which for many exotics nobody has. Finite difference is slower and approximate, but works on anything you can price. That's the tradeoff the practical is pointing at.

### Task C — how big should the bump be?

The book says start at 1e-6 and test what happens as you increase and decrease it. There's a real answer, and it has two failure modes pulling in opposite directions.

In [13]:
args = dict(spot=1.0, strike=1.0, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)
exact = delta_closed_form(OptionType.CALL, **args)

flexes = np.logspace(-14, -0.5, 60)
errors = [abs(delta_finite_difference(OptionType.CALL, **args, spot_flex=h) - exact) / exact
          for h in flexes]

best = flexes[int(np.argmin(errors))]

fig, ax = plt.subplots()
ax.loglog(flexes, errors, color=PRIMARY, linewidth=2)
ax.axvline(best, color=SECONDARY, linestyle="--", linewidth=1.5)
ax.annotate(f"best ≈ {best:.1e}", xy=(best, min(errors)), xytext=(12, 25),
            textcoords="offset points", color=SECONDARY, fontsize=9.5)

ax.annotate("floating-point\ncancellation", xy=(1e-12, 1e-5), fontsize=9.5,
            color=MUTED, ha="center")
ax.annotate("truncation error\n(price curvature)", xy=(1e-2, 1e-5), fontsize=9.5,
            color=MUTED, ha="center")

style_axis(
    ax,
    "Finite-difference bump size: accuracy fails at both ends",
    "Spot bump size h (log scale)",
    "Relative error in delta (log scale)",
    "Too large and you measure the curve, not the tangent. Too small and the two prices differ only in their last bits, and subtracting destroys the signal.",
)
plt.show()

print(f"error at h = 1e-1   {errors[np.argmin(abs(flexes - 1e-1))]:.2e}")
print(f"error at h = 1e-6   {errors[np.argmin(abs(flexes - 1e-6))]:.2e}")
print(f"error at h = 1e-13  {errors[np.argmin(abs(flexes - 1e-13))]:.2e}")

error at h = 1e-1   1.75e-02
error at h = 1e-6   1.09e-11
error at h = 1e-13  1.17e-03


The V shape is the whole story:

- **Right-hand side — truncation error.** A big bump measures a chord across the price curve, not the tangent at a point. The gap is second-order in `h`, so it shrinks as `h²`.
- **Left-hand side — cancellation.** Doubles hold about 16 significant digits. When the two bumped prices agree to 14 of them, subtracting leaves you with 2 digits of signal and 14 of noise, then you divide by a tiny number and amplify it.

The floor sits near `h ≈ √(machine epsilon) ≈ 1e-8`. The book's suggested 1e-6 is comfortably inside the good region.

## Task D — exposure profiles

These profiles are how a trader reads a position (Ch. 9).

In [14]:
spots = np.linspace(0.60, 1.60, 400)
K, r1, r2, sigma = 1.0, 0.0, 0.0, 0.10

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9))

# (a) delta vs spot at several maturities
for T, colour in [(2.0, QUATERNARY), (1.0, PRIMARY), (0.25, TERTIARY), (0.02, SECONDARY)]:
    axes[0, 0].plot(spots, [delta_closed_form(OptionType.CALL, s, K, T, r1, r2, sigma) for s in spots],
                    color=colour, label=f"T = {T}y")
mark_level(axes[0, 0], K, "strike")
axes[0, 0].legend(); as_percent(axes[0, 0])
style_axis(axes[0, 0], "(a) Call delta vs spot", "Spot (CCY2 per CCY1)", "Delta (% of CCY1 notional)",
           "The gradient of each line is gamma. Into expiry it becomes a step from 0% to 100% at the strike.")

# (b) gamma vs spot — the gradient of (a), made explicit
for T, colour in [(2.0, QUATERNARY), (1.0, PRIMARY), (0.25, TERTIARY), (0.02, SECONDARY)]:
    axes[0, 1].plot(spots, [gamma_closed_form(s, K, T, r1, r2, sigma) for s in spots],
                    color=colour, label=f"T = {T}y")
mark_level(axes[0, 1], K, "strike")
axes[0, 1].legend()
style_axis(axes[0, 1], "(b) Gamma vs spot", "Spot (CCY2 per CCY1)", "Gamma (Δdelta per unit spot)",
           "Peak gamma sits at the strike and grows sharply into expiry — the same information as (a)'s steepening.")

# (c) vega vs spot
for T, colour in [(2.0, QUATERNARY), (1.0, PRIMARY), (0.25, TERTIARY), (0.02, SECONDARY)]:
    axes[1, 0].plot(spots, [vega_market(s, K, T, r1, r2, sigma) for s in spots],
                    color=colour, label=f"T = {T}y")
mark_level(axes[1, 0], K, "strike")
axes[1, 0].legend(); as_percent(axes[1, 0], decimals=2)
style_axis(axes[1, 0], "(c) Vega vs spot", "Spot (CCY2 per CCY1)", "Vega (% of CCY1 per 1% vol)",
           "Peak vega also sits at the strike — but unlike gamma it SHRINKS into expiry. That contrast is Chapter 6's main point.")

# (d) value vs volatility, near the money and far away
vols = np.linspace(0.001, 0.60, 300)
for k, label, colour in [(1.00, "K = 1.00 (at the money)", PRIMARY),
                         (1.20, "K = 1.20 (out of the money)", SECONDARY),
                         (1.50, "K = 1.50 (far out)", TERTIARY)]:
    axes[1, 1].plot(vols, [price(OptionType.CALL, 1.0, k, 1.0, r1, r2, v) for v in vols],
                    color=colour, label=label)
axes[1, 1].legend(); as_percent(axes[1, 1], axis="x", decimals=0)
style_axis(axes[1, 1], "(d) Option value vs volatility", "Implied volatility", "Price (CCY2 pips)",
           "The ATM line is almost straight — constant vega. Far strikes curve upward: that convexity is volga.")

plt.tight_layout()
plt.show()

Four things worth naming from those panels:

- **(a) and (b) are the same fact twice.** Gamma is the slope of delta. Compare where (a) is steepest with where (b) peaks.
- **Gamma and vega both peak at the strike** — both come from optionality, which is maximised where the payoff bends.
- **But they move in opposite directions through time.** Peak gamma *grows* into expiry; peak vega *shrinks*. That's why short-dated options are gamma instruments and long-dated ones are vega instruments, with the crossover around two months (Ch. 6).
- **(d) is where volga hides.** The ATM line is near-linear in volatility (vega roughly constant), but the wing strikes curve upward — vega itself rises with volatility out there. That convexity is what the butterfly contract trades (Ch. 12).

### Extreme rates

Practical C asks you to try extreme rate values on the delta and vega charts. The mechanism is the forward.

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
spots = np.linspace(0.55, 1.75, 400)
K, T, sigma = 1.0, 2.0, 0.15

for (r1x, r2x), label, colour in [
    ((0.20, 0.00), "r1 = 20%, r2 = 0%   (forward far below spot)", TERTIARY),
    ((0.00, 0.00), "r1 = r2 = 0%        (forward = spot)", PRIMARY),
    ((0.00, 0.20), "r1 = 0%, r2 = 20%   (forward far above spot)", SECONDARY),
]:
    axes[0].plot(spots, [delta_closed_form(OptionType.CALL, s, K, T, r1x, r2x, sigma) for s in spots],
                 color=colour, label=label)
    axes[1].plot(spots, [vega_market(s, K, T, r1x, r2x, sigma) for s in spots], color=colour)

mark_level(axes[0], K, "strike"); mark_level(axes[1], K, "strike")
axes[0].legend(fontsize=8.5); as_percent(axes[0]); as_percent(axes[1], decimals=2)
style_axis(axes[0], "Delta vs spot under extreme rates", "Spot (CCY2 per CCY1)",
           "Delta (% of CCY1 notional)",
           "High CCY1 rates drag the forward down, so a given spot looks further out of the money.")
style_axis(axes[1], "Vega vs spot under extreme rates", "Spot (CCY2 per CCY1)",
           "Vega (% of CCY1 per 1% vol)",
           "The vega peak tracks the forward, not spot. It shifts away from the strike as the drift grows.")
plt.tight_layout(); plt.show()

The peak sits where the **forward** meets the strike, not where spot does. With a big rate differential over two years the forward is a long way from spot, and the whole profile slides with it. Chapter 6 keeps rates at zero precisely so this doesn't obscure the shape — but a real book has rates in it.

### 3D surfaces

Rotate these. Value, delta and vega over spot × time.

In [16]:
spot_grid = np.linspace(0.70, 1.35, 55)
time_grid = np.linspace(0.02, 2.0, 55)
SS, TT = np.meshgrid(spot_grid, time_grid)
K, r1, r2, sigma = 1.0, 0.0, 0.0, 0.12

surfaces = {
    "Option value (CCY2 pips)": np.vectorize(lambda s, t: price(OptionType.CALL, s, K, t, r1, r2, sigma))(SS, TT),
    "Delta (% of CCY1)": np.vectorize(lambda s, t: delta_closed_form(OptionType.CALL, s, K, t, r1, r2, sigma))(SS, TT),
    "Vega (% per 1% vol)": np.vectorize(lambda s, t: vega_market(s, K, t, r1, r2, sigma))(SS, TT),
}

for name, Z in surfaces.items():
    fig = go.Figure(go.Surface(x=spot_grid, y=time_grid, z=Z, colorscale="Blues",
                               showscale=False, contours={"z": {"show": True, "usecolormap": True}}))
    fig.update_layout(
        scene=dict(xaxis_title="Spot (CCY2/CCY1)", yaxis_title="Time to expiry (years)", zaxis_title=name),
        height=520, title=f"{name} over spot and time — 1.00 strike, 12% vol",
    )
    fig.show()

On the delta surface, follow the ridge from the back (long dated) to the front (near expiry) and watch a gentle slope sharpen into a cliff at the strike. That cliff is what makes expiry day dangerous — Chapter 9's "strikes" section is about managing exactly it, where delta jumps by the full notional as spot crosses the strike.

## Experiments

### Experiment 1 — Which is worth more: 4× the time, or 2× the volatility?

You checked this in notebook 04 with **zero** rates and they were identical. Now put rates in.

**Predict:** still identical, or not?

In [17]:
for (r1x, r2x), label in [((0.0, 0.0), "zero rates"), ((0.01, 0.06), "r1 = 1%, r2 = 6%")]:
    four_t = price(OptionType.CALL, 1.0, 1.0, 4.0, r1x, r2x, 0.10)
    two_v  = price(OptionType.CALL, 1.0, 1.0, 1.0, r1x, r2x, 0.20)
    print(f"{label:22s}  4y/10% = {four_t:.6f}   1y/20% = {two_v:.6f}   gap = {abs(four_t-two_v):.2e}")

zero rates              4y/10% = 0.079656   1y/20% = 0.079656   gap = 0.00e+00
r1 = 1%, r2 = 6%        4y/10% = 0.188602   1y/20% = 0.103466   gap = 8.51e-02


**Result:** identical with zero rates, materially different with rates in.

Volatility and time enter the *distribution's width* only through `σ√T`, so that part is interchangeable. But time also enters through the **drift** `(r₂−r₁)T` and the **discount factor** `e^(−r₂T)`, both of which scale with `T`, not `√T`. Four years of drift is not two years of extra volatility.

### Experiment 2 — Break put–call parity on purpose

**Predict:** you priced a call and a put and found `call − put ≠ F − K`. Before assuming a bug — under what conditions *would* the naive relation hold?

In [18]:
cases = [
    ("r2 = 0, K ≠ F",   dict(spot=1.30, strike=1.20, T=2.0, r_ccy1=0.04, r_ccy2=0.00, sigma=0.12)),
    ("r2 ≠ 0, K = F",   None),
    ("r2 ≠ 0, K ≠ F",   dict(spot=1.30, strike=1.20, T=2.0, r_ccy1=0.01, r_ccy2=0.06, sigma=0.12)),
    ("T → 0",           dict(spot=1.30, strike=1.20, T=1e-8, r_ccy1=0.01, r_ccy2=0.06, sigma=0.12)),
]
for label, a in cases:
    if a is None:
        a = dict(spot=1.30, T=2.0, r_ccy1=0.01, r_ccy2=0.06, sigma=0.12)
        a["strike"] = forward(a["spot"], a["T"], a["r_ccy1"], a["r_ccy2"])
    lhs = price(OptionType.CALL, **a) - price(OptionType.PUT, **a)
    naive = put_call_parity_rhs(a["spot"], a["strike"], a["T"], a["r_ccy1"], a["r_ccy2"], discounted=False)
    print(f"{label:16s}  call-put {lhs:+.8f}   F-K {naive:+.8f}   "
          f"{'HOLDS' if np.isclose(lhs, naive, atol=1e-7) else f'fails by {abs(lhs-naive):.2e}'}")

r2 = 0, K ≠ F     call-put +0.00005125   F-K +0.00005125   HOLDS
r2 ≠ 0, K = F     call-put +0.00000000   F-K +0.00000000   HOLDS
r2 ≠ 0, K ≠ F     call-put +0.20995375   F-K +0.23672219   fails by 2.68e-02
T → 0             call-put +0.10000000   F-K +0.10000000   HOLDS


**Result:** three ways to make the naive version accidentally correct — zero CCY2 rate, strike exactly at the forward (both sides are zero), or zero time to expiry (nothing left to discount). Every other case fails, and the error grows with `r₂ × T`.

The lesson is broader than parity: **any time you compare a premium to a payoff, check they're at the same date.** A present value and a future value are different units.

### Experiment 3 — Does delta really approximate the probability of finishing ITM?

Chapter 6 says delta can be "approximately thought of as the % chance of ending up in-the-money". `N(d₂)` is the actual risk-neutral probability. Delta is `e^(−r₁T)N(d₁)`.

**Predict:** how close are they, and when does the approximation get worse?

In [19]:
rows = []
for T, sigma in [(0.08, 0.10), (1.0, 0.10), (1.0, 0.30), (5.0, 0.30)]:
    for k in [0.90, 1.00, 1.15]:
        d1, d2 = d1_d2(1.0, k, T, 0.0, 0.0, sigma)
        from scipy.stats import norm
        rows.append({
            "T": T, "sigma": sigma, "strike": k,
            "delta = N(d1)": delta_closed_form(OptionType.CALL, 1.0, k, T, 0.0, 0.0, sigma),
            "P(ITM) = N(d2)": norm.cdf(d2),
            "gap": delta_closed_form(OptionType.CALL, 1.0, k, T, 0.0, 0.0, sigma) - norm.cdf(d2),
        })
print(pd.DataFrame(rows).to_string(index=False))

   T  sigma  strike  delta = N(d1)  P(ITM) = N(d2)          gap
0.08    0.1    0.90   9.999077e-01    9.998967e-01 1.095206e-05
0.08    0.1    1.00   5.056417e-01    4.943583e-01 1.128342e-02
0.08    0.1    1.15   4.171059e-07    3.607727e-07 5.633314e-08
1.00    0.1    0.90   8.651178e-01    8.422155e-01 2.290226e-02
1.00    0.1    1.00   5.199388e-01    4.800612e-01 3.987761e-02
1.00    0.1    1.15   8.889041e-02    7.386176e-02 1.502865e-02
1.00    0.3    0.90   6.918854e-01    5.797296e-01 1.121558e-01
1.00    0.3    1.00   5.596177e-01    4.403823e-01 1.192354e-01
1.00    0.3    1.15   3.760494e-01    2.689891e-01 1.070603e-01
5.00    0.3    0.90   6.888073e-01    4.292248e-01 2.595824e-01
5.00    0.3    1.00   6.313422e-01    3.686578e-01 2.626843e-01
5.00    0.3    1.15   5.505557e-01    2.933050e-01 2.572506e-01


**Result:** good for short-dated, low-volatility contracts and increasingly poor as `σ√T` grows. The two differ by exactly the gap between `N(d₁)` and `N(d₂)`, and `d₁ − d₂ = σ√T` — so the approximation degrades precisely as the distribution widens.

This matters in practice. At 1 month and 10% volatility the gap is around one percentage point, so on the desk "25 delta" and "25% chance of expiring ITM" are close enough to use interchangeably. At 1 year and 30% volatility the gap is about **12 points**. At 5 years and 30% it's **26 points** — delta is then telling you something quite different from probability, and treating them as the same thing on long-dated risk will mislead you badly.

In [20]:
K, sigma = 1.0, 0.20
print(f"{'T':>6} {'forward=spot at':>16} {'peak vega spot':>16} {'peak vs strike':>16}")
for T in [0.08, 0.5, 1.0, 3.0, 5.0]:
    spots = np.linspace(0.5, 2.0, 3000)
    vegas = [vega_closed_form(s, K, T, 0.0, 0.0, sigma) for s in spots]
    peak = spots[int(np.argmax(vegas))]
    print(f"{T:>6.2f} {'1.0000':>16} {peak:>16.4f} {peak - K:>+16.4f}")

     T  forward=spot at   peak vega spot   peak vs strike
  0.08           1.0000           1.0017          +0.0017


  0.50           1.0000           1.0102          +0.0102
  1.00           1.0000           1.0202          +0.0202


  3.00           1.0000           1.0617          +0.0617
  5.00           1.0000           1.1052          +0.1052


**Result:** with zero rates the forward equals spot, yet the vega peak drifts *above* the strike as maturity extends — by about **10.5%** at five years.

Why: vega peaks where `d₁ = 0`, and `d₁ = 0` means `ln(S/K) = −(r₂−r₁+σ²/2)T`. With zero rates that leaves the Itō term, so the peak sits at `S = K·e^(σ²T/2)`. Check it against the table: at `T = 5`, `σ = 20%`, that's `e^(0.04×5/2) = e^0.1 = 1.1052`, which is exactly the printed value.

The same `σ²/2` from notebook 04, showing up a third time — in the terminal distribution's drift, in the zero-delta straddle strike (Ch. 8), and now in where vega peaks.

At one month it's 0.17% away and nobody cares. At five years it's over ten percent, and that is why long-dated risk doesn't behave the way short-dated intuition suggests.

## Common misconceptions

**"A put's delta is 25%."**
A put's delta is *negative*. The market quotes it as a positive number and drops the sign in speech — "a ten delta put" has −10% delta. Every formula in this package uses the **true signed value**; `fxds.smile` depends on it, and its docstring says so. This is the single most common source of sign errors in FX derivatives code.

**"ATM means at-the-forward."**
It usually doesn't. In G10 the ATM contract is a **delta-neutral straddle** — strike set so the call and put deltas cancel — which under CCY2 premium sits *above* the forward by `e^(σ²T/2)`. Chapter 7 lists three different contracts that all get called ATM: delta-neutral straddle, ATMF, and ATMS. They are not interchangeable, and which one applies is a per-pair convention.

**"Delta is the probability of finishing in the money."**
It's `N(d₁)`; the probability is `N(d₂)`. They differ by `σ√T`. Fine at 1 month, misleading at 5 years — Experiment 3 measures it.

**"Calls and puts need different risk management."**
Once delta hedged, a call and a put at the same strike and maturity are **the same position** — identical gamma, identical vega, identical everything. That's put–call parity. It's why traders talk about "a 1.3250 strike in EUR50m" and not about calls and puts, and why asking a desk "who does calls and who does puts?" gets you a strange look (Ch. 6).

**"Vega is quoted in the same units the formula produces."**
No. Raw Black-Scholes vega is per 1.0 change in volatility, in CCY2 pips. The market quotes it in CCY1 terms (÷ spot) per **1%** move (÷ 100). Both adjustments, or your number is out by a factor of about 130 in EUR/USD.

**"Premium-included and premium-excluded delta are a rounding detail."**
In CCY1-premium pairs the premium is paid in the currency you're delta hedging, so the hedge has to net it off. That moves the zero-delta straddle strike from *above* the forward to *below* it (Ch. 8). Not implemented in this repo — flagged in `notes/deviations.md` and covered in Chapter 14.

**"Finite difference is just a worse closed form."**
It's a *more general* one. When you reach exotics with no analytic greeks, finite difference is what you have. Practical C makes you build both so the tradeoff is concrete rather than theoretical.

## Check yourself

1. `S = K = 1.0`, `T = 1`, `σ = 10%`, zero rates. Delta comes out at 51.99%, not 50%. Where does the extra 2% come from?
2. A EUR/USD option prices at 0.0250 USD pips with spot at 1.2500 and a EUR20m notional. Give the premium in USD cash, EUR cash and EUR%.
3. Both interest rates rise by 3%. What happens to the forward, and what happens to a vanilla call's price?
4. You compute delta by finite difference with a bump of 1e-15 and get a wildly wrong answer. What went wrong, and which way should you move the bump?
5. Your position is long a 1.3000 EUR/USD call and short a 1.3000 EUR/USD put, same expiry and notional, both delta hedged. What's your net vega?

In [21]:
#@title Answers — run this cell to reveal
from IPython.display import Markdown
Markdown(r'''
**1.** From `d₁ = [ln(S/K) + σ²T/2] / (σ√T)`. With `S = K` the log term vanishes, leaving `d₁ = σ√T/2 = 0.05`, so `N(d₁) = N(0.05) ≈ 0.5199`. It's the Itō correction again. Delta would be exactly 50% at the *delta-neutral straddle strike*, `K = S·e^(σ²T/2) ≈ 1.005`, not at spot — which is the difference between "ATM" and "at-the-forward".

**2.** USD cash `= 0.0250 × 20,000,000 = USD 500,000`. EUR cash `= 500,000 / 1.2500 = EUR 400,000`. EUR% `= 0.0250 / 1.2500 = 2.00%` — which checks out, since `400,000 / 20,000,000 = 2%`.

**3.** The forward is **unchanged** — it depends on the differential `r₂ − r₁`, and both moved together. The call price **falls**, purely through the discount factor `e^(−r₂T)`. Same expected payoff, present valued harder. (Practical C, Task A, Example 3.)

**4.** Floating-point cancellation. The two bumped prices agree to ~15 significant digits, so subtracting leaves almost pure rounding noise, and dividing by 2e-15 amplifies it enormously. Move the bump **up**, to around 1e-6 — see the V-shaped error chart above. The practical floor is near `√(machine epsilon) ≈ 1e-8`.

**5.** **Zero.** Long call + short put at the same strike is a synthetic forward (Ch. 6), and a forward has no optionality — no vega, no gamma. All you hold is a delta position, and you've already hedged it. This is put–call parity turned into a trade.
''')


**1.** From `d₁ = [ln(S/K) + σ²T/2] / (σ√T)`. With `S = K` the log term vanishes, leaving `d₁ = σ√T/2 = 0.05`, so `N(d₁) = N(0.05) ≈ 0.5199`. It's the Itō correction again. Delta would be exactly 50% at the *delta-neutral straddle strike*, `K = S·e^(σ²T/2) ≈ 1.005`, not at spot — which is the difference between "ATM" and "at-the-forward".

**2.** USD cash `= 0.0250 × 20,000,000 = USD 500,000`. EUR cash `= 500,000 / 1.2500 = EUR 400,000`. EUR% `= 0.0250 / 1.2500 = 2.00%` — which checks out, since `400,000 / 20,000,000 = 2%`.

**3.** The forward is **unchanged** — it depends on the differential `r₂ − r₁`, and both moved together. The call price **falls**, purely through the discount factor `e^(−r₂T)`. Same expected payoff, present valued harder. (Practical C, Task A, Example 3.)

**4.** Floating-point cancellation. The two bumped prices agree to ~15 significant digits, so subtracting leaves almost pure rounding noise, and dividing by 2e-15 amplifies it enormously. Move the bump **up**, to around 1e-6 — see the V-shaped error chart above. The practical floor is near `√(machine epsilon) ≈ 1e-8`.

**5.** **Zero.** Long call + short put at the same strike is a synthetic forward (Ch. 6), and a forward has no optionality — no vega, no gamma. All you hold is a delta position, and you've already hedged it. This is put–call parity turned into a trade.


## Where next

You now have both halves of the pricing engine. Confirm they agree:

```bash
pytest tests/test_cross_validation.py -v
```

Two independent routes to the same number is the strongest evidence either one is right — and it's the moment both practicals stop being exercises and become a pricing engine you can build on.

**Notebook 06 — Chapter 6** takes the greeks from a formula to something a desk actually uses: what long gamma feels like, why theta is its price, and how the exposures evolve as expiry approaches.